# Import all needed libraries

In [1]:
import pandas as pd
import numpy as np
import ast
from itertools import combinations
import os

# Get the results dada into a dataset and compute the gap to PHS and gap to OPT metrics

In [2]:
# Define options
family_cols = ['num_warehouses', 'num_customers', 'capacity_distribution']
num_warehouses_options = [2, 3, 4, 5]
num_customers_options = [50, 100, 200, 400]
capacity_distribution_options = ['uniform', 'uneven']

# Initialize an empty list to store dataframes
results_list = []

# Loop through parameter combinations
for num_warehouses in num_warehouses_options:
    for num_customers in num_customers_options:
        for capacity_distribution in capacity_distribution_options:
            # Construct the filename
            filename = f'results/full_tables/full_results_w_{num_warehouses}_c_{num_customers}_d_{capacity_distribution}.csv'
            read_df = pd.read_csv(filename)

            # Keep the last stored result for each policy (in case of multiple runs)
            read_df = read_df.drop_duplicates(subset='Policy', keep='last').reset_index(drop=True)

            # Create a new dataframe
            new_df = pd.DataFrame()
            new_df['num_warehouses'] = [num_warehouses] * len(read_df)
            new_df['num_customers'] = [num_customers] * len(read_df)
            new_df['capacity_distribution'] = [capacity_distribution] * len(read_df)

            # Add all the relevant columns from the read dataframe
            new_df['policy'] = read_df['Policy']
            new_df['avg_reward'] = read_df['MeanReward']
            new_df['std_reward'] = read_df['StdReward']
            new_df['mean_time'] = read_df['MeanTime']
            new_df['std_time'] = read_df['StdTime']
            new_df['all_rewards'] = read_df['AllRewards'].apply(lambda x: eval(x) if isinstance(x, str) else x)
            new_df['all_distance_ranks_n_vectors'] = read_df['AllFacilityProximity'].apply(lambda x: eval(x) if isinstance(x, str) else x)
            new_df['all_distance_ranks_single_vector'] = new_df['all_distance_ranks_n_vectors'].apply(lambda x: [item for sublist in x for item in sublist] if isinstance(x, list) else [])
            new_df['all_times_n_vectors'] = read_df['AllTimes'].apply(lambda x: eval(x) if isinstance(x, str) else x)
            new_df['all_times_single_vector'] = new_df['all_times_n_vectors'].apply(lambda x: [item for sublist in x for item in sublist] if isinstance(x, list) else [])

            # Append to list
            results_list.append(new_df)

# Concatenate all dataframes into a single DataFrame
results = pd.concat(results_list, ignore_index=True)

family_cols = ['num_warehouses', 'num_customers', 'capacity_distribution']

def _as_reward_array(x):
    if isinstance(x, str):
        return np.asarray(ast.literal_eval(x), dtype=float)
    return np.asarray(x, dtype=float)


def _family_of(r):
    return results[(results['num_warehouses'] == r['num_warehouses']) &
                   (results['num_customers'] == r['num_customers']) &
                   (results['capacity_distribution'] == r['capacity_distribution'])]

# Comparison against PHS (Perfect Hindsight Solution)

def _row_stats_phs(r):
    fam = _family_of(r)
    phs_cost = -_as_reward_array(fam[fam['policy'] == 'perfect_hindsight']['all_rewards'].iloc[0])
    cost_pi = -_as_reward_array(r['all_rewards'])

    assert len(cost_pi) == len(phs_cost), "episodes not aligned - cannot pair"
    assert (r['policy'] == 'perfect_hindsight') or ((cost_pi - phs_cost) >= 0).all(), \
        f"PHS not per-episode optimal vs {r['policy']} - episodes misaligned"

    delta = cost_pi - phs_cost
    avg_phs = phs_cost.mean()
    avg_delta, std_delta = delta.mean(), delta.std(ddof=1)
    return pd.Series({
        'delta_phs_vector': delta,
        'avg_delta_phs': avg_delta,
        'std_delta_phs': std_delta,
        'gap_phs_vector': delta / phs_cost * 100.0,
        'avg_gap_phs': avg_delta / avg_phs * 100.0,
        'std_gap_phs': std_delta / avg_phs * 100.0,
    })

results[['delta_phs_vector', 'avg_delta_phs', 'std_delta_phs',
         'gap_phs_vector','avg_gap_phs', 'std_gap_phs']] = results.apply(_row_stats_phs, axis=1)

# Comparison against OPT (Optimal policy)

def _row_stats_evf(r):
    empty = pd.Series({
        'delta_optimal_vector': np.array([]),
        'avg_delta_optimal': np.nan, 'std_delta_optimal': np.nan,
        'avg_gap_optimal': np.nan, 'std_gap_optimal': np.nan,
    })
    fam = _family_of(r)
    evf_rows = fam[fam['policy'] == 'exact_value_function']
    if evf_rows.empty:
        return empty

    evf_cost = -_as_reward_array(evf_rows['all_rewards'].iloc[0])
    cost_pi = -_as_reward_array(r['all_rewards'])
    delta = cost_pi - evf_cost
    avg_evf = evf_cost.mean()
    avg_delta, std_delta = delta.mean(), delta.std(ddof=1)
    return pd.Series({
        'delta_optimal_vector': delta,
        'avg_delta_optimal': avg_delta,
        'std_delta_optimal': std_delta,
        'avg_gap_optimal': avg_delta / avg_evf * 100.0,
        'std_gap_optimal': std_delta / avg_evf * 100.0,
    })

results[['delta_optimal_vector', 'avg_delta_optimal', 'std_delta_optimal',
         'avg_gap_optimal', 'std_gap_optimal']] = results.apply(_row_stats_evf, axis=1)

# quick sanity print (PHS family_gap should be ~0)
_phs_gap = results.loc[results['policy'] == 'perfect_hindsight', 'avg_gap_phs']
print("Consolidation done.", f"results: {len(results)} rows (policy x family).")
if not _phs_gap.empty:
    print(f"  PHS family_gap (should be ~0): {_phs_gap.abs().max():.6f}")

Consolidation done. results: 382 rows (policy x family).
  PHS family_gap (should be ~0): 0.000000


# Tables 5 and E.13 -> Performance comparison tables (against the PHS policy)

In [3]:
# column order
col_policies = ['imitation_learning', 'myopic', 'genetic_programming',
                'linear_value_function_approximation', 'deep_q_networks',
                'point_estimate_lookahead', 'distributional_estimate_lookahead',
                'linear_programming_heuristic',
                'parameterized_lookahead_approximation', 'proximal_policy_optimization']

_header = r"""    \multicolumn{3}{c|}{\textbf{Instance description}}
    & \multicolumn{1}{c|}{\textbf{\gls{pfa}}} & \multicolumn{2}{c|}{\textbf{\gls{cfa}}} & \multicolumn{2}{c|}{\textbf{\gls{vfa}}} & \multicolumn{2}{c|}{\textbf{\gls{dla}}} & \multicolumn{3}{c}{\textbf{Hybrid}}   \\
    \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{\gls{il}} & \textbf{\gls{myo}} & \textbf{\gls{gp}} & \textbf{\gls{lvfa}} & \textbf{\gls{dqn}} & \textbf{\gls{pel}} & \textbf{\gls{del}} & \textbf{\gls{lph}} & \textbf{\gls{pla}} & \textbf{PPO}\\
"""

_header_plain = r"""    \multicolumn{3}{c|}{\textbf{Instance description}} %& \multicolumn{8}{c|}{\textbf{Cost (\% above perfect hindsight)}} \\
    & \multicolumn{1}{c|}{\textbf{\gls{pfa}}} & \multicolumn{2}{c|}{\textbf{\gls{cfa}}} & \multicolumn{2}{c|}{\textbf{\gls{vfa}}} & \multicolumn{2}{c|}{\textbf{\gls{dla}}} & \multicolumn{3}{c}{\textbf{Hybrid}}   \\
    \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{\gls{il}} & \textbf{\gls{myo}} & \textbf{\gls{gp}} & \textbf{\gls{lvfa}} & \textbf{\gls{dqn}} & \textbf{\gls{pel}} & \textbf{\gls{del}} & \textbf{\gls{lph}} & \textbf{\gls{pla}} & \textbf{PPO}\\
"""

# Functions for the per-family summary rows (average, min, max) and for the table body

def _cell_gap(w, c, d, policy):
    row = results[(results.num_warehouses == w) & (results.num_customers == c) &
                  (results.capacity_distribution == d) & (results.policy == policy)]
    return float(row['avg_gap_phs'].iloc[0]), float(row['std_gap_phs'].iloc[0])

def _avg_gap(policy):
    sub = results[results.policy == policy]
    return float(sub['avg_gap_phs'].mean()), float(sub['avg_gap_phs'].std(ddof=1))

def _minmax_gap(policy, kind):
    sub = results[results.policy == policy]
    idx = sub['avg_gap_phs'].idxmin() if kind == 'min' else sub['avg_gap_phs'].idxmax()
    row = sub.loc[idx]
    return float(row['avg_gap_phs']), float(row['std_gap_phs'])

def _family_rows(with_ci):
    rows = ""
    for num_customers in num_customers_options:
        rows += f"        \\multirow{{8}}{{*}}{{{num_customers}}} "
        for num_warehouses in num_warehouses_options:
            for i, cap in enumerate(capacity_distribution_options):
                if i == 0:
                    rows += f"& \\multirow{{2}}{{*}}{{{num_warehouses}}} & {cap.capitalize()} & "
                else:
                    rows += f"& & {cap.capitalize()} & "
                vals = {p: _cell_gap(num_warehouses, num_customers, cap, p) for p in col_policies}
                best = min(vals, key=lambda p: vals[p][0])
                cells = []
                for p in col_policies:
                    m, h = vals[p]
                    s = f"{m:.2f} ({h:.2f})" if with_ci else f"{m:.2f}"
                    cells.append(f"\\textbf{{{s}}}" if p == best else s)
                rows += " & ".join(cells) + " \\\\\n"
        if num_customers != num_customers_options[-1]:
            rows += "        \\hline\n"
    return rows

def _summary_row(label, values, with_ci):
    best = min(values, key=lambda p: values[p][0])
    cells = []
    for p in col_policies:
        m, h = values[p]
        s = f"{m:.2f} ({h:.2f})" if with_ci else f"{m:.2f}"
        cells.append(f"\\textbf{{{s}}}" if p == best else s)
    return f"\\multicolumn{{3}}{{c|}}{{{label}}} & " + " & ".join(cells) + " \\\\\n"

def _build_gap_table_plain():
    avg = {p: _avg_gap(p) for p in col_policies}
    mn = {p: _minmax_gap(p, 'min') for p in col_policies}
    mx = {p: _minmax_gap(p, 'max') for p in col_policies}
    t = r"""
\begin{table}[H]
    \small
    \centering
    \caption{Policy comparison across all families of instances: percentage gap between each policy's total realized cost and the \gls{phs}'s total cost, aggregated over each family's episodes. For each instance family, every policy is evaluated on the same demand realizations. The bottom summary rows report, for each policy, the average of its per-family gaps (equal weight per family), and the minimum and maximum per-family gap. The best policy in each row is in bold.}
    \label{tab:policy_comparison_performance}
    \begin{tabular}{ccc|S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]S[table-format=2.2]}
    \toprule
""" + _header_plain + r"""        \toprule
""" + _family_rows(with_ci=False) \
    + "        \\midrule\n" \
        + _summary_row("Average", avg, with_ci=False) \
        + _summary_row("Min", mn, with_ci=False) \
        + _summary_row("Max", mx, with_ci=False) \
        + r"""    \bottomrule
    \end{tabular}
\end{table}
"""
    return t

def _build_gap_table_std():
    t = r"""
\begin{table}[!b]
    \centering
    \rotatebox{90}{%
        \begin{minipage}{0.94\textheight}
        \centering
        \caption{\scriptsize Policy comparison across all families of instances: percentage gap between each policy's total realized cost and the \gls{phs}'s total cost, aggregated over each family's episodes, with a paired dispersion measure. For each instance family, every policy is evaluated on the same demand realizations. Each cell reports the average gap to the \gls{phs}, and the value in parentheses is the standard deviation of the paired per-episode difference between the policy's cost and the \gls{phs}'s cost within that family, scaled by the family's mean \gls{phs} cost. The best policy in each row is in bold.}
        \label{tab:policy_comparison_performance_ci}
        \resizebox{0.96\textheight}{!}{%
        \setlength{\tabcolsep}{8pt}
        \begin{tabular}{ccc|c|cc|cc|cc|ccc}
    \toprule
""" + _header + r"""        \toprule
""" + _family_rows(with_ci=True) \
        + r"""        \bottomrule

        \end{tabular}%
        }
        \end{minipage}%
    }
\end{table}
"""
    return t

print("% ===== MAIN per-family sum-ratio gap table (Average/Min/Max) =====")
print(_build_gap_table_plain())
print("\n% ===== per-family sum-ratio gap table WITH dispersion (Average/Min/Max) =====")
print(_build_gap_table_std())

% ===== MAIN per-family sum-ratio gap table (Average/Min/Max) =====

\begin{table}[H]
    \small
    \centering
    \caption{Policy comparison across all families of instances: percentage gap between each policy's total realized cost and the \gls{phs}'s total cost, aggregated over each family's episodes. For each instance family, every policy is evaluated on the same demand realizations. The bottom summary rows report, for each policy, the average of its per-family gaps (equal weight per family), and the minimum and maximum per-family gap. The best policy in each row is in bold.}
    \label{tab:policy_comparison_performance}
    \begin{tabular}{ccc|S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]S[table-format=2.2]}
    \toprule
    \multicolumn{3}{c|}{\textbf{Instance description}} %& \multicolumn{8}{c|}{\textbf{Cost (\% above perfect hindsight)}} \\
    & \mu

# Table B.7 -> Performance comparison tables (against the OPT policy)

In [4]:
# Get all the rows with a computed optimal policy (exact_value_function) for the per-family gap table
_results_optimal = results[results['avg_gap_optimal'].notna()]

# Functions for the per-family summary rows (average, min, max) and for the table body for the optimal policy comparison
def _cell_gap_optimal(w, c, d, policy):
    row = _results_optimal[(_results_optimal.num_warehouses == w) &
                        (_results_optimal.num_customers == c) &
                        (_results_optimal.capacity_distribution == d) &
                        (_results_optimal.policy == policy)]
    if row.empty:
        return None
    return float(row['avg_gap_optimal'].iloc[0]), float(row['std_gap_optimal'].iloc[0])

def _avg_gap_optimal(policy):
    sub = _results_optimal[_results_optimal.policy == policy]
    if sub.empty:
        return float('nan'), float('nan')
    return float(sub['avg_gap_optimal'].mean()), float(sub['avg_gap_optimal'].std(ddof=1))

def _minmax_gap_optimal(policy, kind):
    sub = _results_optimal[_results_optimal.policy == policy]
    if sub.empty:
        return float('nan'), float('nan')
    idx = sub['avg_gap_optimal'].idxmin() if kind == 'min' else sub['avg_gap_optimal'].idxmax()
    row = sub.loc[idx]
    return float(row['avg_gap_optimal']), float(row['std_gap_optimal'])

def _family_rows_optimal():
    rows = ""
    for num_customers in num_customers_options:
        rows += f"        \\multirow{{8}}{{*}}{{{num_customers}}} "
        for num_warehouses in num_warehouses_options:
            for i, cap in enumerate(capacity_distribution_options):
                if i == 0:
                    rows += f"& \\multirow{{2}}{{*}}{{{num_warehouses}}} & {cap.capitalize()} & "
                else:
                    rows += f"& & {cap.capitalize()} & "
                vals = {p: _cell_gap_optimal(num_warehouses, num_customers, cap, p) for p in col_policies}
                cells = []
                for p in col_policies:
                    v = vals[p]
                    s = "" if v is None else f"{v[0]:.2f}"
                    cells.append(s)
                rows += " & ".join(cells) + " \\\\\n"
        if num_customers != num_customers_options[-1]:
            rows += "        \\hline\n"
    return rows

def _build_gap_table_optimal_plain():
    avg = {p: _avg_gap_optimal(p) for p in col_policies}
    mn = {p: _minmax_gap_optimal(p, 'min') for p in col_policies}
    mx = {p: _minmax_gap_optimal(p, 'max') for p in col_policies}
    t = r"""
\begin{table}[H]
    \small
    \centering
     \caption{Policy comparison across all families of instances: percentage gap between each policy's total realized cost and the total cost of the optimal policy (solved by backward dynamic programming), aggregated over each family's episodes. Only families with a computed optimal policy are populated; the remaining cells are left blank. For each populated instance family, every policy is evaluated on the same demand realizations. The bottom summary rows report, for each policy, the average of its per-family gaps (equal weight per family), and the minimum and maximum per-family gap, computed only over the populated families.}
    \label{tab:policy_comparison_performance_evf}
    \begin{tabular}{ccc|S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]S[table-format=2.2]}
    \toprule
""" + _header_plain + r"""        \toprule
""" + _family_rows_optimal() \
    + "        \\midrule\n" \
        + _summary_row("Average", avg, with_ci=False) \
        + _summary_row("Min", mn, with_ci=False) \
        + _summary_row("Max", mx, with_ci=False) \
        + r"""    \bottomrule
    \end{tabular}
\end{table}
"""
    return t

print("% ===== OPT per-family sum-ratio gap table (Average/Min/Max) =====")
print(_build_gap_table_optimal_plain())

% ===== OPT per-family sum-ratio gap table (Average/Min/Max) =====

\begin{table}[H]
    \small
    \centering
     \caption{Policy comparison across all families of instances: percentage gap between each policy's total realized cost and the total cost of the optimal policy (solved by backward dynamic programming), aggregated over each family's episodes. Only families with a computed optimal policy are populated; the remaining cells are left blank. For each populated instance family, every policy is evaluated on the same demand realizations. The bottom summary rows report, for each policy, the average of its per-family gaps (equal weight per family), and the minimum and maximum per-family gap, computed only over the populated families.}
    \label{tab:policy_comparison_performance_evf}
    \begin{tabular}{ccc|S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]S[ta

# Table F.14 -> Runtime comparison table

In [5]:
# Get the ordering and labels for the policies in the table
policies = ['imitation_learning', 'myopic', 'genetic_programming', 'linear_value_function_approximation', 'deep_q_networks', 'point_estimate_lookahead', 'distributional_estimate_lookahead', 'linear_programming_heuristic','parameterized_lookahead_approximation', 'proximal_policy_optimization']
policies_dict = {'myopic': 'MYO', 'imitation_learning': 'IL', 'point_estimate_lookahead': 'PEL', 'distributional_estimate_lookahead': 'DEL', 'parameterized_lookahead_approximation': 'CFA-DLA', 'linear_programming_heuristic': 'LPH', 'linear_programming_exact': 'LPE', 'proximal_policy_optimization': 'PPO'}

# Policies whose time is presented as microseconds (short runtimes) vs milliseconds (long runtimes)
short_time_policies = ['myopic', 'genetic_programming', 'linear_value_function_approximation', 'deep_q_networks', 'imitation_learning', 'proximal_policy_optimization']

def _time_robust_stats(raw_times, policy):
    times_array = np.asarray(raw_times, dtype=float)
    if policy in short_time_policies:
        times_array = times_array / 1_000  # -> microseconds
    else:
        times_array = times_array / 1_000_000  # -> milliseconds
    return (
        float(np.median(times_array)),
        float(np.percentile(times_array, 80)),
        float(np.percentile(times_array, 90)),
        float(np.percentile(times_array, 95)),
        float(np.percentile(times_array, 99)),
        float(times_array.max()),
    )

latex_table = r"""
\begin{table}[!b]
    \centering
    \rotatebox{90}{%
        \begin{minipage}{0.93\textheight}
        \centering
        \caption{\scriptsize Policy comparison across all families of instances: online runtime per order, in milliseconds, robust to outliers. Each cell reports the median runtime over that family's episodes, with the 95th percentile runtime in parentheses. Columns marked with * correspond to very low runtime values and are multiplied by $10^3$; the values in these columns can therefore be read directly as microseconds. The fastest policy in each row (by median) is in bold.}
        \label{tab:policy_comparison_time_robust}
        \resizebox{0.96\textheight}{!}{%
        %\small
        \setlength{\tabcolsep}{8pt}
        \begin{tabular}{ccc|c|cc|cc|cc|cccc}
        \toprule
        \multicolumn{3}{c|}{\textbf{Instance description}} %& \multicolumn{8}{c|}{\textbf{Cost (\% above perfect hindsight)}} \\
    & \multicolumn{1}{c|}{\textbf{\gls{pfa}}} & \multicolumn{2}{c|}{\textbf{\gls{cfa}}} & \multicolumn{2}{c|}{\textbf{\gls{vfa}}} & \multicolumn{2}{c|}{\textbf{\gls{dla}}} & \multicolumn{3}{c}{\textbf{Hybrid}}   \\
    \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{\gls{il}\textsuperscript{*}} & \textbf{\gls{myo}\textsuperscript{*}} & \textbf{\gls{gp}\textsuperscript{*}} & \textbf{\gls{lvfa}\textsuperscript{*}} & \textbf{\gls{dqn}\textsuperscript{*}} & \textbf{\gls{pel}} & \textbf{\gls{del}} & \textbf{\gls{lph}} & \textbf{\gls{pla}} & \textbf{\gls{ppo}\textsuperscript{*}}\\
        \toprule
"""

# Loop through all combinations of num_customers, num_warehouses, and capacity_distribution to populate the LaTeX table
for num_customers in num_customers_options:
    latex_table += f"        \\multirow{{8}}{{*}}{{{num_customers}}} "
    for num_warehouses in num_warehouses_options:
        for i, capacity_distribution in enumerate(capacity_distribution_options):
            if i == 0:
                latex_table += f"& \\multirow{{2}}{{*}}{{{num_warehouses}}} & {capacity_distribution.capitalize()} & "
            else:
                latex_table += f"& & {capacity_distribution.capitalize()} & "

            # Filter the DataFrame for the current configuration
            df_filtered = results[
                (results["num_customers"] == num_customers) &
                (results["num_warehouses"] == num_warehouses) &
                (results["capacity_distribution"] == capacity_distribution)
            ]

            # Extract the median / p80 / p95 / p99 / worst runtime for each policy, from the raw per-episode times
            medians, p80s, p90s, p95s, p99s, worsts, short_medians = [], [], [], [], [], [], []
            for policy in policies:
                raw_times = df_filtered.loc[df_filtered['policy'] == policy, 'all_times_single_vector'].values[0]
                median, p80, p90, p95, p99, worst = _time_robust_stats(raw_times, policy)
                raw_times_arr = np.asarray(raw_times, dtype=float)
                worst3_idx = np.argsort(raw_times_arr)[-3:][::-1]
                #print(f"[check] customers={num_customers} warehouses={num_warehouses} capacity={capacity_distribution} policy={policy} worst3_idx={worst3_idx.tolist()} worst3_vals={raw_times_arr[worst3_idx].tolist()}")
                medians.append(median)
                p80s.append(p80)
                p90s.append(p90)
                p95s.append(p95)
                p99s.append(p99)
                worsts.append(worst)
                if policy in short_time_policies:
                    short_medians.append(median)

            # Find the min median in short_medians, knowing that's the relevant comparison group
            min_median = min(short_medians)

            # Format the cells, highlighting the minimum median in bold; p95 and worst shown in parentheses
            # Although several percentiles are computed, only p95 is shown in the table for brevity
            formatted_cells = [
                f"\\textbf{{{median:.2f}}} ({p95:.2f})" if median == min_median
                else f"{median:.2f} ({p95:.2f})"
                for median, p80, p90, p95, p99, worst in zip(medians, p80s, p90s, p95s, p99s, worsts)
            ]
            
            latex_table += " & ".join(formatted_cells) + r" \\" + "\n"
    latex_table += "        \\bottomrule\n"

# Close the LaTeX table structure
latex_table += r"""
    \end{tabular}
            }%
        \end{minipage}
        }
    \end{table}
"""

# Print the LaTeX table string
print(latex_table)


\begin{table}[!b]
    \centering
    \rotatebox{90}{%
        \begin{minipage}{0.93\textheight}
        \centering
        \caption{\scriptsize Policy comparison across all families of instances: online runtime per order, in milliseconds, robust to outliers. Each cell reports the median runtime over that family's episodes, with the 95th percentile runtime in parentheses. Columns marked with * correspond to very low runtime values and are multiplied by $10^3$; the values in these columns can therefore be read directly as microseconds. The fastest policy in each row (by median) is in bold.}
        \label{tab:policy_comparison_time_robust}
        \resizebox{0.96\textheight}{!}{%
        %\small
        \setlength{\tabcolsep}{8pt}
        \begin{tabular}{ccc|c|cc|cc|cc|cccc}
        \toprule
        \multicolumn{3}{c|}{\textbf{Instance description}} %& \multicolumn{8}{c|}{\textbf{Cost (\% above perfect hindsight)}} \\
    & \multicolumn{1}{c|}{\textbf{\gls{pfa}}} & \multicolumn{2}{c|}{\

# Table 4 -> Pairwise t-tests between policies

In [6]:
matrix_policies = ['imitation_learning', 'myopic', 'genetic_programming',
                   'linear_value_function_approximation', 'deep_q_networks',
                   'point_estimate_lookahead', 'distributional_estimate_lookahead',
                   'linear_programming_heuristic',
                   'parameterized_lookahead_approximation', 'proximal_policy_optimization']

fam_keys = list(results.groupby(['num_warehouses', 'num_customers', 'capacity_distribution']).groups.keys())

# Family's mean PHS cost, cached once per family (same quantity avg_gap_phs divides by).
avg_phs_by_family = {}
for key in fam_keys:
    fam_phs_rows = results[(results['num_warehouses'] == key[0]) &
                            (results['num_customers'] == key[1]) &
                            (results['capacity_distribution'] == key[2]) &
                            (results['policy'] == 'perfect_hindsight')]
    avg_phs_by_family[key] = _as_reward_array(fam_phs_rows['all_rewards'].iloc[0]).mean() * -1

rows = []
for _, r in results[results['policy'].isin(matrix_policies)].iterrows():
    key = (r['num_warehouses'], r['num_customers'], r['capacity_distribution'])
    family_id = f"w{key[0]}_c{key[1]}_{key[2]}"
    #perf = np.asarray(r['delta_phs_vector']) / avg_phs_by_family[key] * 100.0
    #use the reward vector simply
    perf = np.asarray(r['all_rewards']) * -1
    for instance_id, p in enumerate(perf):
        rows.append({
            'family_id': family_id,
            'instance_id': instance_id,
            'policy_id': r['policy'],
            'performance': p,
        })

data = pd.DataFrame(rows)

#Order by policy_id, family_id, instance_id for consistency with the tables above
data = data.sort_values(by=['policy_id', 'family_id', 'instance_id']).reset_index(drop=True)

#Make the order random but dont delete rows (ascending in family_order and instance for same policies but not for other policies

n_fam = data['family_id'].nunique()
n_pol = data['policy_id'].nunique()
inst_per_fam_pol = data.groupby(['family_id', 'policy_id']).size().unique()
print(f"data: {len(data)} rows "
      f"({n_fam} families x {n_pol} policies x {int(list(inst_per_fam_pol)[0])} simulation episodes per family-policy)")

data.to_csv("results/full_tables/data_for_stats.csv", index=False)

data: 64000 rows (32 families x 10 policies x 200 simulation episodes per family-policy)


In [7]:
#Libraries for statistical tests and respective documentation links

from scipy.stats import ttest_rel
#https://docs.scipy.org/doc/scipy-1.15.2/reference/generated/scipy.stats.ttest_rel.html

from statsmodels.stats.multitest import multipletests
#https://www.statsmodels.org/stable/generated/statsmodels.stats.multitest.multipletests.html

CONFIDENCE_LEVEL = 0.95
print_tables = False

alpha = 1 - CONFIDENCE_LEVEL
policy_order = matrix_policies
n_pol = len(policy_order)
pairs = list(combinations(range(n_pol), 2))
families = sorted(data['family_id'].unique())

# Get the data in wide format for easier pairwise comparisons
wide_data = data.pivot(index=['family_id', 'instance_id'], columns='policy_id', values='performance')

# Initialize matrices to store the number of wins and draws for each policy pair
ttest_wins = np.zeros((n_pol, n_pol), dtype=int)
ttest_draws = np.zeros((n_pol, n_pol), dtype=int)
ttest_posthoc_tables = {}

# Perform paired t-tests for each family and each policy pair and store the results
for family_id in families:
    fam_wide = wide_data.loc[family_id]

    pvals, diffs = [], []
    for ia, ib in pairs:
        a, b = policy_order[ia], policy_order[ib]
        _, p = ttest_rel(fam_wide[a].values, fam_wide[b].values)
        pvals.append(p)
        diffs.append(float(np.mean(fam_wide[a].values - fam_wide[b].values)))

    reject, p_adj, _, _ = multipletests(pvals, alpha=alpha, method='holm')

    if print_tables:
        pmat = pd.DataFrame(np.nan, index=policy_order, columns=policy_order)
        for (ia, ib), p in zip(pairs, p_adj):
            pmat.iloc[ia, ib] = pmat.iloc[ib, ia] = p
        ttest_posthoc_tables[family_id] = pmat

    for (ia, ib), p, diff, rej in zip(pairs, p_adj, diffs, reject):
        if rej:
            if diff < 0:
                ttest_wins[ia, ib] += 1
            else:
                ttest_wins[ib, ia] += 1
        else:
            ttest_draws[ia, ib] += 1
            ttest_draws[ib, ia] += 1

# sanity check: row-wins + col-wins + draws == number of families, for every pair
for ia, ib in pairs:
    assert ttest_wins[ia, ib] + ttest_wins[ib, ia] + ttest_draws[ia, ib] == len(families)

print(f"Paired t-test posthoc done: {len(families)} families x {len(pairs)} pairwise comparisons, "
      f"alpha={alpha:.2f} (Holm-corrected within each family).")


Paired t-test posthoc done: 32 families x 45 pairwise comparisons, alpha=0.05 (Holm-corrected within each family).


In [8]:
# Policy labels for the LaTeX table, in the same order as policy_order
matrix_labels = {'myopic': 'MYO', 'imitation_learning': 'IL', 'genetic_programming': 'GP',
                  'point_estimate_lookahead': 'PEL', 'distributional_estimate_lookahead': 'DEL',
                  'linear_value_function_approximation': 'LVFA', 'deep_q_networks': 'DQN',
                  'linear_programming_heuristic': 'LPH', 'linear_programming_exact': 'LPE',
                  'parameterized_lookahead_approximation': 'PLA', 'proximal_policy_optimization': 'PPO'}

labels = [matrix_labels[p] for p in policy_order]
n_families = len(families)
col_fmt = "l|" + "c" * n_pol

tt = r"""
\begin{table}[h!]
    \centering
    \scriptsize
    \renewcommand{\arraystretch}{1.35}
    \setlength{\tabcolsep}{4pt}
    \caption{Pairwise comparison of policies' performance across the 32 families of instances. For each pair of policies and each family, a paired $t$-test is conducted on the 200 paired episode-level costs within that family, with $p$-values Holm-corrected across the 55 pairwise comparisons within the family. Each cell reports, out of the 32 families, the number of families in which the row policy has a significantly smaller cost than the column policy, the number in which the column policy has a significantly smaller cost, and the number in which no significant difference is found (row wins / column wins / draws, at $\alpha=0.05$), so that the three values in each cell sum to 32. The \textbf{Net (W$-$L)} row reports, for each column policy, the difference between its total wins and total losses across all 320 pairwise family comparisons (9 opposing policies $\times$ 32 families). This value would only reach $+288$ or $-288$ in the absence of any draws; in practice, draws reduce the maximum attainable magnitude, as reflected, for example, in the \gls{myo} policy's value of $-278$ despite losing every non-drawn comparison.}
    \label{tab:pairwise_ttest_family_summary}
    \begin{tabular}{l|ccccccccccc}
    \toprule
    \textbf{Row $\backslash$ Col} & """ + " & ".join(f"\\textbf{{{x}}}" for x in labels) + r""" \\
    \midrule
"""
# Generate the table body: for each row policy, report the wins/losses/draws against each column policy
for ia, a in enumerate(policy_order):
    cells = [f"\\textbf{{{labels[ia]}}}"]
    for ib, b in enumerate(policy_order):
        if ia == ib:
            cells.append("")
        else:
            cells.append(f"{ttest_wins[ia, ib]}/{ttest_wins[ib, ia]}/{ttest_draws[ia, ib]}")
    tt += "    " + " & ".join(cells) + r" \\" + "\n"

# summary row: for each column policy, aggregate a/b/c (wins / losses / draws) against all other policies
col_summary = []
for j in range(n_pol):
    losses_j = sum(ttest_wins[i, j] for i in range(n_pol) if i != j)
    wins_j = sum(ttest_wins[j, i] for i in range(n_pol) if i != j)
    draws_j = sum(ttest_draws[i, j] for i in range(n_pol) if i != j)
    col_summary.append((wins_j, losses_j, draws_j))

tt += "    \\midrule\n"
net_cells = [r"\textbf{Net (W$-$L)}"] + [f"{wins - losses:+d}" for wins, losses, draws in col_summary]
tt += "    " + " & ".join(net_cells) + r" \\" + "\n"

tt += r"""    \bottomrule
    \end{tabular}
\end{table}
"""
print("% ===== paired t-test win/loss/draw summary matrix =====")
print(tt)

% ===== paired t-test win/loss/draw summary matrix =====

\begin{table}[h!]
    \centering
    \scriptsize
    \renewcommand{\arraystretch}{1.35}
    \setlength{\tabcolsep}{4pt}
    \caption{Pairwise comparison of policies' performance across the 32 families of instances. For each pair of policies and each family, a paired $t$-test is conducted on the 200 paired episode-level costs within that family, with $p$-values Holm-corrected across the 55 pairwise comparisons within the family. Each cell reports, out of the 32 families, the number of families in which the row policy has a significantly smaller cost than the column policy, the number in which the column policy has a significantly smaller cost, and the number in which no significant difference is found (row wins / column wins / draws, at $\alpha=0.05$), so that the three values in each cell sum to 32. The \textbf{Net (W$-$L)} row reports, for each column policy, the difference between its total wins and total losses across all 320

# Table D.11 -> LVFA final weights

In [9]:
num_customers_options = [50, 100, 200, 400]
num_warehouses_options = [2, 3, 4, 5]
capacity_distribution_options = ['uniform', 'uneven']

# Read the theta.csv file
theta_df = pd.read_csv('training/linear_value_function_approximation_training/theta.csv')

# Define the LaTeX table structure
latex_table = r"""
\begin{table}[H]
    \centering
    \scriptsize
    \caption{\gls{lvfa} final weights ($\theta$) across all families of instances.}
    \label{tab:lvfa_theta_all}
    \begin{tabular}{ccc|S[table-format=4.2]S[table-format=4.2]S[table-format=4.2]S[table-format=4.2]S[table-format=4.2]}
        \toprule
        \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{$\theta_A$} & \textbf{$\theta_B$} & \textbf{$\theta_C$} & \textbf{$\theta_D$} & \textbf{$\theta_E$} \\
        \midrule
"""

# Loop through the customer, warehouse, and capacity options to fill the table
for num_customers in num_customers_options:
    latex_table += f"        \\multirow{{8}}{{*}}{{{num_customers}}} "
    for num_warehouses in num_warehouses_options:
        for i, capacity_distribution in enumerate(capacity_distribution_options):
            if i == 0:
                latex_table += f"& \\multirow{{2}}{{*}}{{{num_warehouses}}} & {capacity_distribution.capitalize()} & "
            else:
                latex_table += f"& & {capacity_distribution.capitalize()} & "

            # Filter the DataFrame for the current configuration
            df_filtered = theta_df[
                (theta_df["num_customers"] == num_customers) &
                (theta_df["num_warehouses"] == num_warehouses) &
                (theta_df["capacity_distribution"] == capacity_distribution)
            ]

            # pad with blanks up to the 5-column max (num_warehouses_options tops out at 5)
            MAX_THETA_COLS = 5
            if not df_filtered.empty:
                theta = df_filtered.iloc[0]['theta']
                theta_values = [f"{round(float(t), 2)}" for t in theta.split(',')]
            else:
                theta_values = []
            theta_cells = theta_values + [""] * (MAX_THETA_COLS - len(theta_values))
            latex_table += " & ".join(theta_cells) + " \\\\\n"
    latex_table += "        \\hline\n"
latex_table += "        \\bottomrule\n"

# Close the LaTeX table structure
latex_table += r"""
    \end{tabular}
\end{table}
"""

# Print the LaTeX table string
print(latex_table)


\begin{table}[H]
    \centering
    \scriptsize
    \caption{\gls{lvfa} final weights ($\theta$) across all families of instances.}
    \label{tab:lvfa_theta_all}
    \begin{tabular}{ccc|S[table-format=4.2]S[table-format=4.2]S[table-format=4.2]S[table-format=4.2]S[table-format=4.2]}
        \toprule
        \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{$\theta_A$} & \textbf{$\theta_B$} & \textbf{$\theta_C$} & \textbf{$\theta_D$} & \textbf{$\theta_E$} \\
        \midrule
        \multirow{8}{*}{50} & \multirow{2}{*}{2} & Uniform & 57.01 & 56.6 &  &  &  \\
& & Uneven & 72.92 & 29.8 &  &  &  \\
& \multirow{2}{*}{3} & Uniform & 55.81 & 31.21 & 52.75 &  &  \\
& & Uneven & 73.99 & 24.82 & 27.22 &  &  \\
& \multirow{2}{*}{4} & Uniform & 42.56 & 43.05 & 37.39 & 37.54 &  \\
& & Uneven & 63.06 & 54.87 & 18.26 & -5.84 &  \\
& \multirow{2}{*}{5} & Uniform & 36.17 & 36.1 & 36.25 & 36.34 & 46.94 \\
& & Uneven & 59.15 & 45.83 & 26.49 & 12.98 & 29.75 \\
        \hli

# Table D.12 -> PLA parameter tuning

In [10]:
# Tested parameters for PLA
param_options = [1.0, 1.02, 1.04, 1.06, 1.08, 1.10]

pla_df = pd.read_csv('training/parameterized_lookahead_approximation_training/parameterized_lookahead_approximation_all_params.csv')

latex_table = r"""
\begin{table}[H]
    \scriptsize
    \centering
    \caption{\gls{pla} mean cost over 50 simulation episodes per tested $\theta$, and selected $\theta$ across all families of instances.}
    \label{tab:pla_theta_sensitivity}
    \begin{tabular}{ccc|S[table-format=5.2]S[table-format=5.2]S[table-format=5.2]S[table-format=5.2]S[table-format=5.2]S[table-format=5.2]|S[table-format=1.2]}
    \toprule
    \multicolumn{3}{c|}{\textbf{Instance description}} & \multicolumn{6}{c|}{\textbf{Mean cost per tested $\theta$}} & \\
    \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{$\theta=1.00$} & \textbf{$\theta=1.02$} & \textbf{$\theta=1.04$} & \textbf{$\theta=1.06$} & \textbf{$\theta=1.08$} & \textbf{$\theta=1.10$} & \textbf{Selected $\theta$} \\
        \toprule
"""

# Loop through the customer, warehouse, and capacity options to fill the table
for num_customers in num_customers_options:
    latex_table += f"        \\multirow{{8}}{{*}}{{{num_customers}}} "
    for num_warehouses in num_warehouses_options:
        for i, capacity_distribution in enumerate(capacity_distribution_options):
            if i == 0:
                latex_table += f"& \\multirow{{2}}{{*}}{{{num_warehouses}}} & {capacity_distribution.capitalize()} & "
            else:
                latex_table += f"& & {capacity_distribution.capitalize()} & "

            # Filter the DataFrame for the current configuration
            df_filtered = pla_df[
                (pla_df["num_customers"] == num_customers) &
                (pla_df["num_warehouses"] == num_warehouses) &
                (pla_df["capacity_distribution"] == capacity_distribution)
            ]

            # Extract the mean reward for each tested theta
            rewards = []
            for param in param_options:
                reward = df_filtered.loc[df_filtered['param'] == param, 'mean_reward'].values[0]
                rewards.append(reward)

            # Best theta = highest mean reward (least negative)
            best_reward = max(rewards)
            best_param = param_options[rewards.index(best_reward)]

            # The reward values are negative, so we negate them to show positive costs in the table. The best reward is highlighted in bold.
            formatted_rewards = [
                f"\\textbf{{{-reward:.2f}}}" if reward == best_reward else f"{-reward:.2f}"
                for reward in rewards
            ]

            latex_table += " & ".join(formatted_rewards) + f" & {best_param:.2f} " + r" \\" + "\n"
    latex_table += "        \\hline\n"

# Close the LaTeX table structure
latex_table += r"""
    \end{tabular}
\end{table}
"""

# Print the LaTeX table string
print(latex_table)


\begin{table}[H]
    \scriptsize
    \centering
    \caption{\gls{pla} mean cost over 50 simulation episodes per tested $\theta$, and selected $\theta$ across all families of instances.}
    \label{tab:pla_theta_sensitivity}
    \begin{tabular}{ccc|S[table-format=5.2]S[table-format=5.2]S[table-format=5.2]S[table-format=5.2]S[table-format=5.2]S[table-format=5.2]|S[table-format=1.2]}
    \toprule
    \multicolumn{3}{c|}{\textbf{Instance description}} & \multicolumn{6}{c|}{\textbf{Mean cost per tested $\theta$}} & \\
    \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{$\theta=1.00$} & \textbf{$\theta=1.02$} & \textbf{$\theta=1.04$} & \textbf{$\theta=1.06$} & \textbf{$\theta=1.08$} & \textbf{$\theta=1.10$} & \textbf{Selected $\theta$} \\
        \toprule
        \multirow{8}{*}{50} & \multirow{2}{*}{2} & Uniform & \textbf{3267.67} & 3269.50 & 3270.93 & 3277.07 & 3282.80 & 3288.08 & 1.00  \\
& & Uneven & \textbf{3485.22} & 3486.65 & 3486.22 & 3487.86 & 3487

# Table A.6 -> Initial capacity of each facility

In [11]:
from env import InventoryEnv

latex_table = r"""
\begin{table}[H]
    \centering
    \scriptsize
    \caption{Initial facility capacity across all families of instances.}
    \label{tab:facility_initial_capacity_all}
    \begin{tabular}{ccc|S[table-format=3.0]S[table-format=3.0]S[table-format=3.0]S[table-format=3.0]S[table-format=3.0]}
        \toprule
        \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{$A$} & \textbf{$B$} & \textbf{$C$} & \textbf{$D$} & \textbf{$E$} \\
        \midrule
"""

# Loop through the customer, warehouse, and capacity options to fill the table
for num_customers in num_customers_options:
    latex_table += f"        \\multirow{{8}}{{*}}{{{num_customers}}} "
    for num_warehouses in num_warehouses_options:
        for i, capacity_distribution in enumerate(capacity_distribution_options):
            if i == 0:
                latex_table += f"& \\multirow{{2}}{{*}}{{{num_warehouses}}} & {capacity_distribution.capitalize()} & "
            else:
                latex_table += f"& & {capacity_distribution.capitalize()} & "

            MAX_FACILITY_COLS = 5
            env = InventoryEnv(num_warehouses, num_customers, capacity_distribution)

            # Get the initial capacities for the warehouses and format them for the LaTeX table
            capacity_values = [str(v) for v in env.warehouses_initial_capacity]
            capacity_cells = capacity_values + [""] * (MAX_FACILITY_COLS - len(capacity_values))
            latex_table += " & ".join(capacity_cells) + " \\\\\n"
    latex_table += "        \\hline\n"
latex_table += "        \\bottomrule\n"

# Close the LaTeX table structure
latex_table += r"""
    \end{tabular}
\end{table}
"""

# Print the LaTeX table string
print(latex_table)


\begin{table}[H]
    \centering
    \scriptsize
    \caption{Initial facility capacity across all families of instances.}
    \label{tab:facility_initial_capacity_all}
    \begin{tabular}{ccc|S[table-format=3.0]S[table-format=3.0]S[table-format=3.0]S[table-format=3.0]S[table-format=3.0]}
        \toprule
        \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{$A$} & \textbf{$B$} & \textbf{$C$} & \textbf{$D$} & \textbf{$E$} \\
        \midrule
        \multirow{8}{*}{50} & \multirow{2}{*}{2} & Uniform & 25 & 25 &  &  &  \\
& & Uneven & 35 & 15 &  &  &  \\
& \multirow{2}{*}{3} & Uniform & 17 & 17 & 16 &  &  \\
& & Uneven & 25 & 15 & 10 &  &  \\
& \multirow{2}{*}{4} & Uniform & 13 & 13 & 12 & 12 &  \\
& & Uneven & 20 & 15 & 10 & 5 &  \\
& \multirow{2}{*}{5} & Uniform & 10 & 10 & 10 & 10 & 10 \\
& & Uneven & 16 & 12 & 10 & 7 & 5 \\
        \hline
        \multirow{8}{*}{100} & \multirow{2}{*}{2} & Uniform & 50 & 50 &  &  &  \\
& & Uneven & 70 & 30 &  &  & 

# Table D.8 -> Training times (in minutes) across all policies requiring training

In [12]:
def generate_training_time_table(
    policies,
    num_warehouses_options=(2, 3, 4, 5),
    num_customers_options=(50, 100, 200, 400),
    capacity_distribution_options=('uniform', 'uneven'),
    policies_dict=None,
    caption=None,
    label='tab:policy_training_time',
):
    
    default_labels = {
        'myopic': 'MYO', 'imitation_learning': 'IL', 'genetic_programming': 'GP',
        'linear_value_function_approximation': 'LVFA', 'deep_q_networks': 'DQN',
        'point_estimate_lookahead': 'PEL', 'distributional_estimate_lookahead': 'DEL',
        'linear_programming_heuristic': 'LPH', 'linear_programming_exact': 'LPE',
        'parameterized_lookahead_approximation': 'PLA',
        'proximal_policy_optimization': 'PPO', 'exact_value_function': 'OPT',
    }
    labels = {**default_labels, **(policies_dict or {})}

    # Load each policy's training times into {(w, c, d): training_time}
    times_by_policy = {}
    for policy in policies:
        csv_path = f'training/{policy}_training/{policy}_training_times.csv'
        if not os.path.exists(csv_path):
            print(f"[warning] no training-time CSV for '{policy}' at {csv_path}; "
                  f"leaving its column blank in the table.")
            times_by_policy[policy] = {}
            continue
        df = pd.read_csv(csv_path)
        times_by_policy[policy] = {
            (int(r.num_warehouses), int(r.num_customers), r.capacity_distribution): float(r.training_time)
            for r in df.itertuples()
        }

    # Column width (integer digits) per policy, from its own observed max value
    col_format = {}
    for policy in policies:
        values = list(times_by_policy[policy].values())
        max_int_digits = len(str(int(max(values)))) if values else 1
        col_format[policy] = max(max_int_digits, 1)

    header_cols = " & ".join(f"\\textbf{{\\gls{{{labels.get(p, p).lower()}}}}}" for p in policies)
    col_spec = "ccc|" + "".join(f"S[table-format={col_format[p]}.2]" for p in policies)

    if caption is None:
        caption = ("Training time (in minutes) of each policy across all families of instances.")

    latex_table = "\\begin{table}[H]\n"
    latex_table += "    \\scriptsize\n    \\centering\n"
    latex_table += f"    \\caption{{{caption}}}\n"
    latex_table += f"    \\label{{{label}}}\n"
    latex_table += f"    \\begin{{tabular}}{{{col_spec}}}\n"
    latex_table += "    \\toprule\n"
    latex_table += (f"    \\textbf{{$|\\OrderSet|$}} & \\textbf{{$|\\FacilitySet|$}} & "
                     f"\\textbf{{Capacity}} & {header_cols} \\\\\n")
    latex_table += "    \\midrule\n"

    rows_per_family = len(num_warehouses_options) * len(capacity_distribution_options)
    for f_idx, num_customers in enumerate(num_customers_options):
        latex_table += f"        \\multirow{{{rows_per_family}}}{{*}}{{{num_customers}}} "
        for num_warehouses in num_warehouses_options:
            for i, capacity_distribution in enumerate(capacity_distribution_options):
                if i == 0:
                    latex_table += f"& \\multirow{{{len(capacity_distribution_options)}}}{{*}}{{{num_warehouses}}} & {capacity_distribution.capitalize()} & "
                else:
                    latex_table += f"& & {capacity_distribution.capitalize()} & "

                cells = []
                for policy in policies:
                    value = times_by_policy[policy].get((num_warehouses, num_customers, capacity_distribution))
                    cells.append(f"{value:.2f}" if value is not None else "")
                latex_table += " & ".join(cells) + r" \\" + "\n"
        if f_idx < len(num_customers_options) - 1:
            latex_table += "        \\hline\n"

    latex_table += "        \\bottomrule\n"
    latex_table += "    \\end{tabular}\n\\end{table}\n"
    return latex_table

# Provide the policies to include (only those with training time data available)
training_time_policies = ['imitation_learning', 'genetic_programming', 'linear_value_function_approximation', 'deep_q_networks', 'parameterized_lookahead_approximation', 'proximal_policy_optimization', 'exact_value_function']

print(generate_training_time_table(training_time_policies))

\begin{table}[H]
    \scriptsize
    \centering
    \caption{Training time (in minutes) of each policy across all families of instances.}
    \label{tab:policy_training_time}
    \begin{tabular}{ccc|S[table-format=1.2]S[table-format=1.2]S[table-format=1.2]S[table-format=2.2]S[table-format=1.2]S[table-format=2.2]S[table-format=2.2]}
    \toprule
    \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{\gls{il}} & \textbf{\gls{gp}} & \textbf{\gls{lvfa}} & \textbf{\gls{dqn}} & \textbf{\gls{pla}} & \textbf{\gls{ppo}} & \textbf{\gls{opt}} \\
    \midrule
        \multirow{8}{*}{50} & \multirow{2}{*}{2} & Uniform & 0.43 & 0.47 & 0.27 & 7.46 & 0.52 & 12.22 & 0.00 \\
& & Uneven & 0.42 &  & 0.27 & 7.44 & 0.53 & 12.07 & 0.00 \\
& \multirow{2}{*}{3} & Uniform & 0.44 &  & 0.32 & 7.54 & 0.74 & 12.20 & 0.00 \\
& & Uneven & 0.44 &  & 0.31 & 7.53 & 0.75 & 12.25 & 0.00 \\
& \multirow{2}{*}{4} & Uniform & 0.44 &  & 0.36 & 7.42 & 0.97 & 12.16 & 0.01 \\
& & Uneven & 0.45 &  & 0